In [ ]:
"""

Title: "Scalable Neural Safety Certification via Monotonicity"
Code for: Learing a neural barrier certificate for Network of coupled oscillators (9000D) example.
Author: Amirreza Alavi
Contact: seyedamirreza.alavi@colorado.edu
Affiliation: University of Colorado Boulder
Date: 04/2026

"""

In [ ]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers, constraints, initializers
import matplotlib.pyplot as plt
from numpy import linalg as LA
from scipy import stats
from functools import partial
%matplotlib inline
import numpy as np
from scipy.optimize import fsolve
from tensorflow.keras.layers import Input, Dense, Concatenate, Add
from tensorflow.keras import Model
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model
import time
from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D  
from math import inf
import heapq
from dataclasses import dataclass
import matplotlib.patches as mpatches
from tensorflow.keras import layers, constraints, Model

In [ ]:
# System Definition

t1 = time.perf_counter()

def phi_kuramoto(theta):

    theta = tf.convert_to_tensor(theta, dtype=tf.float32)
    squeeze = (theta.shape.rank == 1)
    if squeeze:
        theta = theta[None, :]  

    N = tf.shape(theta)[1]

    diff = theta[:, None, :] - theta[:, :, None]      
    s = tf.sin(diff)                                  

    mask = tf.ones((N, N), dtype=tf.float32) - tf.eye(N, dtype=tf.float32)  
    s = s * mask[None, :, :]                          

    phi = tf.reduce_sum(s, axis=-1)                 

    if squeeze:
        phi = phi[0]
    return phi


def f_kuramoto(theta, nu):

    K=0.0012
    tau=0.1
    #Omega_val=0.01
    
    theta = tf.convert_to_tensor(theta, dtype=tf.float32)
    nu    = tf.convert_to_tensor(nu, dtype=tf.float32)

    squeeze = (theta.shape.rank == 1)
    if squeeze:
        theta = theta[None, :]  

    N = tf.shape(theta)[1]

    if nu.shape.rank == 1:
        nu = nu[None, :]      

    # Omega = tf.fill((1, N), tf.cast(Omega_val, tf.float32)) 
    # Omega = tf.broadcast_to(Omega, tf.shape(theta))

    K   = tf.cast(K, tf.float32)
    tau = tf.cast(tau, tf.float32)

    phi = phi_kuramoto(theta)  

    theta_next = theta + (tau * nu) + ( (tau * K) / tf.cast(N, tf.float32) )*phi 

    pi = tf.constant(np.pi, dtype=tf.float32)

    theta_next = tf.clip_by_value(
        theta_next,
        clip_value_min=0.0,
        clip_value_max=0.25 * pi
    )

    if squeeze:
        theta_next = theta_next[0] 

    return theta_next

In [ ]:
# System Inductive evaluation functions

def f_kuramoto_0_km1(theta0, modelK, k):

    theta = tf.convert_to_tensor(theta0, dtype=tf.float32)
    if theta.shape.rank == 1:
        theta = tf.expand_dims(theta, 0)

    B, N = theta.shape

    traj = tf.TensorArray(tf.float32, size=k, element_shape=(B, N))
    for t in tf.range(k):
        traj = traj.write(t, theta)
        nu = kappa(theta, modelK)
        theta = f_kuramoto(theta, nu)

    return traj.stack()



def f_kuramoto_k(theta0, modelK, k):

    theta = tf.convert_to_tensor(theta0, dtype=tf.float32)
    squeeze = (theta.shape.rank == 1)
    if squeeze:
        theta = tf.expand_dims(theta, 0)

    for _ in tf.range(k):
        nu = kappa(theta, modelK)
        theta = f_kuramoto(theta, nu)

    return theta[0] if squeeze else theta

In [ ]:
# Sampled points for a coarse gridding

N=9000

thetain_bar = np.ones((1,N))*(np.pi/16)
thetain_und = np.zeros((1,N))
thetauns_und = np.eye(N)*(np.pi/8)
thetauns_bar = np.ones((1,N))*(np.pi/4)
thetai_bar= np.ones((1,N))*(np.pi/8)
thetai_und= np.zeros((1,N))

In [ ]:
# Neural Barrier and Controller Definition


class ScalarMonotoneNN(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None,
                 output_activation="linear", output_dim=1):
        super().__init__()

        if activation is None:
            activation = tf.keras.layers.LeakyReLU(alpha=0.1)

        self.hidden_layers = []
        for h in hidden_dims:
            self.hidden_layers.append(
                layers.Dense(
                    h,
                    activation=activation,
                    use_bias=True,
                    kernel_constraint=constraints.NonNeg()
                )
            )

        self.out_layer = layers.Dense(
            output_dim,
            activation=output_activation,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

    def call(self, x):
        z = x
        for layer in self.hidden_layers:
            z = layer(z)
        return self.out_layer(z)


class MaxAffineBranch(layers.Layer):

    def __init__(self, m_i, hidden_dims=(200, 200), activation=None,
                 output_activation="linear", output_dim=1):
        super().__init__()

        self.affine = layers.Dense(
            m_i,
            activation=None,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation,
            output_dim=output_dim
        )

    def call(self, x):
        z = self.affine(x)                            
        g = tf.reduce_max(z, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                        
        return y


class InputMaxBranch(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None,
                 output_activation="linear", output_dim=1):
        super().__init__()

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation,
            output_dim=output_dim
        )

    def call(self, x):
        g = tf.reduce_max(x, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                        
        return y


class ThreeBranchMinMonotoneNetwork(Model):

    def __init__(self,
                 input_dim,
                 m1,
                 m2,
                 output_dim=1,
                 hidden_dims1=(200, 200),
                 hidden_dims2=(200, 200),
                 hidden_dims3=(200, 200),
                 activation=None,
                 output_activation="linear"):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim

        self.branch1 = InputMaxBranch(
            hidden_dims=hidden_dims1,
            activation=activation,
            output_activation=output_activation,
            output_dim=output_dim
        )

        self.branch2 = MaxAffineBranch(
            m_i=m1,
            hidden_dims=hidden_dims2,
            activation=activation,
            output_activation=output_activation,
            output_dim=output_dim
        )

        self.branch3 = MaxAffineBranch(
            m_i=m2,
            hidden_dims=hidden_dims3,
            activation=activation,
            output_activation=output_activation,
            output_dim=output_dim
        )

    def call(self, x):
        y1 = self.branch1(x)   
        y2 = self.branch2(x)   
        y3 = self.branch3(x)   

        Y = tf.stack([y1, y2, y3], axis=1)   
        y = tf.reduce_min(Y, axis=1)        

        return y


# Barrier: B: R^N -> R
modelB = ThreeBranchMinMonotoneNetwork(
    input_dim=N,
    m1=int(0.2*N),
    m2=int(0.1*N),
    output_dim=1,
    hidden_dims1=(200, 200),
    hidden_dims2=(200, 200),
    hidden_dims3=(200, 200),
    activation=tf.keras.layers.LeakyReLU(alpha=0.1),
    output_activation="linear"
)


# Controller: kappa: R^N -> R^N
modelK = ThreeBranchMinMonotoneNetwork(
    input_dim=N,
    m1=int(0.2*N),
    m2=int(0.1*N),
    output_dim=N,
    hidden_dims1=(200, 200),
    hidden_dims2=(200, 200),
    hidden_dims3=(200, 200),
    activation=tf.keras.layers.LeakyReLU(alpha=0.1),
    output_activation="linear"
)

def kappa(theta, modelK):
    raw_u = modelK(theta)
    return (np.pi / 4.0) * tf.tanh(raw_u)

In [ ]:
# Training Neural Safe Controller

epochs = 100000
opt1 = tf.keras.optimizers.Adam(1e-2)
opt2 = tf.keras.optimizers.Adam(1e-2)


gamma = -2
eta = 2
k=2


for i in range(epochs):
    with tf.GradientTape() as tape0, tf.GradientTape() as tape1:

        loss_B = tf.reduce_sum(tf.nn.relu(tf.subtract(1.0,1.0)))
        loss_B_termination = tf.reduce_sum(tf.nn.relu(tf.subtract(1.0,1.0)))

        loss_B += tf.square(tf.nn.relu(tf.reduce_max(modelB(tf.reshape(f_kuramoto_0_km1(thetain_bar, modelK, k), (-1, tf.shape(thetain_bar)[-1]))) - gamma)))
        loss_B += tf.square(tf.nn.relu(eta - tf.reduce_min(modelB(thetauns_und))))
        loss_B += tf.math.sign(tf.nn.relu(-(tf.reduce_max( modelB(tf.reshape(f_kuramoto_0_km1(thetai_und, modelK, k), (-1, tf.shape(thetai_und)[-1]))) ) - 0.0)))  * (tf.nn.relu(modelB(f_kuramoto_k(thetai_bar, modelK, k)) -  0.0))

        
        loss_B_termination += tf.nn.relu(tf.reduce_max(modelB(tf.reshape(f_kuramoto_0_km1(thetain_bar, modelK, k), (-1, tf.shape(thetain_bar)[-1]))) - (-0.1)))
        loss_B_termination += tf.nn.relu(0.1 - tf.reduce_min(modelB(thetauns_und)))
        loss_B_termination += tf.math.sign(tf.nn.relu(-(tf.reduce_max( modelB(tf.reshape(f_kuramoto_0_km1(thetai_und, modelK, k), (-1, tf.shape(thetai_und)[-1]))) ) - 0.0)))  * (tf.nn.relu(modelB(f_kuramoto_k(thetai_bar, modelK, k)) -  0.0))

        third_loss_B = tf.math.sign(tf.nn.relu(-(tf.reduce_max( modelB(tf.reshape(f_kuramoto_0_km1(thetai_und, modelK, k), (-1, tf.shape(thetai_und)[-1]))) ) - 0.0)))  * (tf.nn.relu(modelB(f_kuramoto_k(thetai_bar, modelK, k)) -  0.0))
        third_loss_B_val = third_loss_B.numpy().item()

                                
        grad_B = tape0.gradient(loss_B, modelB.trainable_variables)
        grad_K = tape1.gradient(loss_B, modelK.trainable_variables)
                
        if  loss_B_termination == 0   and i>1:
            print('Safety Verified')
            break
            
        if (i // 2) % 2 == 0:
            clipgrad_B = [tf.clip_by_norm(g, 2) for g in grad_B]
            opt1.apply_gradients(zip(clipgrad_B, modelB.trainable_variables))
        else:
            if grad_K is not None:
                clipgrad_K = [tf.clip_by_norm(g, 2) for g in grad_K]
                opt2.apply_gradients(zip(clipgrad_K, modelK.trainable_variables))
            
        if i%5 == 1:
            print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(tf.squeeze(f_kuramoto_0_km1(thetain_bar, modelK, k)))).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(thetauns_und)).numpy():.4f}, "
            f"Third condition = {third_loss_B_val:.4f}, "
            f"LossB = {loss_B_termination.numpy().item():.4f}"
            )


    
t2 = time.perf_counter()
tot = t2 - t1
print(f'Total runtime: {tot}s')
print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(tf.squeeze(f_kuramoto_0_km1(thetain_bar, modelK, k)))).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(thetauns_und)).numpy():.4f}, "
            f"Third condition = {third_loss_B_val:.4f}, "
            f"LossB = {loss_B_termination.numpy().item():.4f}"
            )